# Hierarchical Plant Disease Detection System
## Stage 2: Disease Classification per Plant Species

This notebook implements the second stage of a hierarchical plant disease detection pipeline:
1. **Data reorganization** by plant species
2. **Training** separate MobileNetV3-based classifiers for each plant
3. **Inference pipeline** that combines plant and disease classification

**Architecture:** MobileNetV3Large with transfer learning  
**Plants:** Apple, Tomato, Potato, Corn, Pepper  
**Optimization:** Kaggle GPU runtime with mixed precision training

## 1. Setup and Configuration

In [ ]:
# Import libraries
import os
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import pickle
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import MobileNetV3Large
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

# Enable mixed precision for faster training on GPU
from tensorflow.keras import mixed_precision
policy = mixed_precision.Policy('mixed_float16')
mixed_precision.set_global_policy(policy)
print(f"Mixed precision enabled: {policy.name}")

In [ ]:
# Configuration
class Config:
    # Paths
    DATASET_PATH = '/kaggle/input/plantvillage-dataset/PlantVillage'  # Adjust to your dataset path
    REORGANIZED_PATH = '/kaggle/working/reorganized_dataset'
    MODELS_PATH = '/kaggle/working/disease_models'
    PLANT_CLASSIFIER_PATH = '/kaggle/input/plant-classifier/plant_classifier.h5'  # Your stage 1 model
    
    # Training parameters
    IMG_SIZE = (224, 224)
    BATCH_SIZE = 32
    EPOCHS = 30
    LEARNING_RATE = 0.001
    VALIDATION_SPLIT = 0.2
    
    # Plant species and their diseases
    PLANT_DISEASES = {
        'Apple': [
            'Apple___Black_rot',
            'Apple___Cedar_apple_rust',
            'Apple___Apple_scab',
            'Apple___healthy'
        ],
        'Tomato': [
            'Tomato___Bacterial_spot',
            'Tomato___Early_blight',
            'Tomato___Late_blight',
            'Tomato___Leaf_Mold',
            'Tomato___Septoria_leaf_spot',
            'Tomato___Spider_mites Two-spotted_spider_mite',
            'Tomato___Target_Spot',
            'Tomato___Tomato_Yellow_Leaf_Curl_Virus',
            'Tomato___Tomato_mosaic_virus',
            'Tomato___healthy'
        ],
        'Potato': [
            'Potato___Early_blight',
            'Potato___Late_blight',
            'Potato___healthy'
        ],
        'Corn': [
            'Corn___Cercospora_leaf_spot Gray_leaf_spot',
            'Corn___Common_rust',
            'Corn___Northern_Leaf_Blight',
            'Corn___healthy'
        ],
        'Pepper': [
            'Pepper__bell___Bacterial_spot',
            'Pepper__bell___healthy'
        ]
    }

config = Config()

# Create necessary directories
os.makedirs(config.MODELS_PATH, exist_ok=True)
os.makedirs(config.REORGANIZED_PATH, exist_ok=True)

## 2. Data Reorganization

In [ ]:
def reorganize_dataset_by_plant(config):
    """
    Reorganizes the PlantVillage dataset from Plant___Disease structure
    to separate directories per plant species for individual training.
    
    Structure:
    reorganized_dataset/
        Apple/
            train/
                Black_rot/
                Cedar_apple_rust/
                Apple_scab/
                healthy/
            val/
                ...
        Tomato/
            ...
    """
    print("=" * 70)
    print("REORGANIZING DATASET BY PLANT SPECIES")
    print("=" * 70)
    
    dataset_path = Path(config.DATASET_PATH)
    reorganized_path = Path(config.REORGANIZED_PATH)
    
    # Clear existing reorganized data
    if reorganized_path.exists():
        shutil.rmtree(reorganized_path)
    
    stats = {}
    
    for plant, diseases in config.PLANT_DISEASES.items():
        print(f"\n{plant}:")
        plant_stats = {'total': 0, 'train': 0, 'val': 0, 'diseases': {}}
        
        for disease_folder in diseases:
            # Extract disease name (remove plant prefix)
            disease_name = disease_folder.split('___')[-1].replace('__bell___', '')
            
            source_dir = dataset_path / disease_folder
            
            if not source_dir.exists():
                print(f"  ⚠️  Warning: {disease_folder} not found")
                continue
            
            # Get all images
            images = list(source_dir.glob('*.jpg')) + list(source_dir.glob('*.JPG'))
            images += list(source_dir.glob('*.png')) + list(source_dir.glob('*.PNG'))
            
            if len(images) == 0:
                print(f"  ⚠️  Warning: No images found in {disease_folder}")
                continue
            
            # Shuffle and split
            np.random.shuffle(images)
            split_idx = int(len(images) * (1 - config.VALIDATION_SPLIT))
            train_images = images[:split_idx]
            val_images = images[split_idx:]
            
            # Create directories and copy files
            for split, img_list in [('train', train_images), ('val', val_images)]:
                dest_dir = reorganized_path / plant / split / disease_name
                dest_dir.mkdir(parents=True, exist_ok=True)
                
                for img in img_list:
                    shutil.copy2(img, dest_dir / img.name)
            
            plant_stats['diseases'][disease_name] = {
                'train': len(train_images),
                'val': len(val_images),
                'total': len(images)
            }
            plant_stats['total'] += len(images)
            plant_stats['train'] += len(train_images)
            plant_stats['val'] += len(val_images)
            
            print(f"  ✓ {disease_name}: {len(train_images)} train, {len(val_images)} val")
        
        stats[plant] = plant_stats
        print(f"  → Total: {plant_stats['total']} images ({plant_stats['train']} train, {plant_stats['val']} val)")
    
    # Save statistics
    with open(reorganized_path / 'dataset_stats.json', 'w') as f:
        json.dump(stats, f, indent=2)
    
    print("\n" + "=" * 70)
    print("REORGANIZATION COMPLETE")
    print("=" * 70)
    
    return stats

# Execute reorganization
dataset_stats = reorganize_dataset_by_plant(config)

## 3. Model Architecture Factory

In [ ]:
def create_disease_classifier(num_classes, plant_name):
    """
    Creates a MobileNetV3Large-based disease classifier with transfer learning.
    
    Architecture:
    - MobileNetV3Large backbone (pretrained on ImageNet)
    - Global Average Pooling
    - Dense layer with dropout for regularization
    - Output layer with softmax activation
    
    Args:
        num_classes: Number of disease classes for this plant
        plant_name: Name of the plant (for model naming)
    
    Returns:
        Compiled Keras model
    """
    # Load pretrained MobileNetV3Large
    base_model = MobileNetV3Large(
        input_shape=(*config.IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    
    # Freeze base model for initial training
    base_model.trainable = False
    
    # Build classifier head
    inputs = keras.Input(shape=(*config.IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu', name='dense_features')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', dtype='float32', name='predictions')(x)
    
    model = keras.Model(inputs, outputs, name=f'{plant_name}_disease_classifier')
    
    # Compile with Adam optimizer
    model.compile(
        optimizer=optimizers.Adam(learning_rate=config.LEARNING_RATE),
        loss='categorical_crossentropy',
        metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_accuracy')]
    )
    
    return model, base_model

# Test model creation
test_model, _ = create_disease_classifier(4, 'Apple')
test_model.summary()

## 4. Data Generators with Augmentation

In [ ]:
def create_data_generators(plant_name, config):
    """
    Creates training and validation data generators with augmentation.
    
    Training augmentation:
    - Random rotation (±15°)
    - Width/height shift (±10%)
    - Zoom (±20%)
    - Horizontal flip
    - Shear transformation
    - Brightness adjustment
    
    All images are rescaled to [0, 1] range.
    """
    train_dir = Path(config.REORGANIZED_PATH) / plant_name / 'train'
    val_dir = Path(config.REORGANIZED_PATH) / plant_name / 'val'
    
    # Training data generator with augmentation
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.2,
        horizontal_flip=True,
        shear_range=0.1,
        brightness_range=[0.8, 1.2],
        fill_mode='nearest'
    )
    
    # Validation data generator (only rescaling)
    val_datagen = ImageDataGenerator(rescale=1./255)
    
    # Create generators
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=config.IMG_SIZE,
        batch_size=config.BATCH_SIZE,
        class_mode='categorical',
        shuffle=True,
        seed=42
    )
    
    val_generator = val_datagen.flow_from_directory(
        val_dir,
        target_size=config.IMG_SIZE,
        batch_size=config.BATCH_SIZE,
        class_mode='categorical',
        shuffle=False
    )
    
    return train_generator, val_generator

# Test generator creation
train_gen, val_gen = create_data_generators('Apple', config)
print(f"Training samples: {train_gen.samples}")
print(f"Validation samples: {val_gen.samples}")
print(f"Class indices: {train_gen.class_indices}")

## 5. Training Pipeline

In [ ]:
def train_disease_classifier(plant_name, config, fine_tune=True):
    """
    Trains a disease classifier for a specific plant species.
    
    Training strategy:
    1. Phase 1: Train only the classifier head (base frozen)
    2. Phase 2 (optional): Fine-tune top layers of base model
    
    Args:
        plant_name: Name of the plant species
        config: Configuration object
        fine_tune: Whether to fine-tune base model after initial training
    
    Returns:
        Trained model, training history, class indices
    """
    print("\n" + "=" * 70)
    print(f"TRAINING DISEASE CLASSIFIER: {plant_name.upper()}")
    print("=" * 70)
    
    # Create data generators
    train_gen, val_gen = create_data_generators(plant_name, config)
    num_classes = len(train_gen.class_indices)
    
    print(f"\nClasses ({num_classes}): {list(train_gen.class_indices.keys())}")
    print(f"Training samples: {train_gen.samples}")
    print(f"Validation samples: {val_gen.samples}")
    
    # Create model
    model, base_model = create_disease_classifier(num_classes, plant_name)
    
    # Callbacks
    model_path = Path(config.MODELS_PATH) / f'{plant_name}_disease_classifier.h5'
    callbacks = [
        EarlyStopping(
            monitor='val_accuracy',
            patience=5,
            restore_best_weights=True,
            verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=3,
            min_lr=1e-7,
            verbose=1
        ),
        ModelCheckpoint(
            str(model_path),
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]
    
    # Phase 1: Train classifier head
    print("\n" + "-" * 70)
    print("PHASE 1: Training classifier head (base model frozen)")
    print("-" * 70)
    
    history1 = model.fit(
        train_gen,
        epochs=config.EPOCHS // 2,
        validation_data=val_gen,
        callbacks=callbacks,
        verbose=1
    )
    
    # Phase 2: Fine-tuning (optional)
    if fine_tune:
        print("\n" + "-" * 70)
        print("PHASE 2: Fine-tuning top layers of base model")
        print("-" * 70)
        
        # Unfreeze top layers of base model
        base_model.trainable = True
        for layer in base_model.layers[:-30]:  # Freeze all but last 30 layers
            layer.trainable = False
        
        # Recompile with lower learning rate
        model.compile(
            optimizer=optimizers.Adam(learning_rate=config.LEARNING_RATE / 10),
            loss='categorical_crossentropy',
            metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_accuracy')]
        )
        
        history2 = model.fit(
            train_gen,
            epochs=config.EPOCHS // 2,
            validation_data=val_gen,
            callbacks=callbacks,
            verbose=1
        )
        
        # Combine histories
        history = {
            key: history1.history[key] + history2.history[key]
            for key in history1.history.keys()
        }
    else:
        history = history1.history
    
    # Save class indices
    class_indices_path = Path(config.MODELS_PATH) / f'{plant_name}_class_indices.json'
    with open(class_indices_path, 'w') as f:
        json.dump(train_gen.class_indices, f, indent=2)
    
    print(f"\n✓ Model saved: {model_path}")
    print(f"✓ Class indices saved: {class_indices_path}")
    
    return model, history, train_gen.class_indices, val_gen

def plot_training_history(history, plant_name):
    """Plots training and validation metrics."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Accuracy
    axes[0].plot(history['accuracy'], label='Train Accuracy')
    axes[0].plot(history['val_accuracy'], label='Val Accuracy')
    axes[0].set_title(f'{plant_name} - Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Loss
    axes[1].plot(history['loss'], label='Train Loss')
    axes[1].plot(history['val_loss'], label='Val Loss')
    axes[1].set_title(f'{plant_name} - Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{config.MODELS_PATH}/{plant_name}_training_history.png', dpi=150)
    plt.show()

## 6. Train All Plant Disease Classifiers

In [ ]:
# Train classifiers for all plants
trained_models = {}
training_histories = {}

for plant_name in config.PLANT_DISEASES.keys():
    model, history, class_indices, val_gen = train_disease_classifier(
        plant_name, 
        config, 
        fine_tune=True
    )
    
    trained_models[plant_name] = {
        'model': model,
        'class_indices': class_indices,
        'val_generator': val_gen
    }
    training_histories[plant_name] = history
    
    # Plot training history
    plot_training_history(history, plant_name)
    
    # Clear memory
    tf.keras.backend.clear_session()

print("\n" + "=" * 70)
print("ALL MODELS TRAINED SUCCESSFULLY")
print("=" * 70)

## 7. Model Evaluation

In [ ]:
def evaluate_model(model, val_generator, plant_name, class_indices):
    """Evaluates model performance and generates classification report."""
    print(f"\n{'=' * 70}")
    print(f"EVALUATION: {plant_name.upper()}")
    print("=" * 70)
    
    # Get predictions
    val_generator.reset()
    predictions = model.predict(val_generator, verbose=1)
    y_pred = np.argmax(predictions, axis=1)
    y_true = val_generator.classes
    
    # Reverse class indices mapping
    class_labels = {v: k for k, v in class_indices.items()}
    target_names = [class_labels[i] for i in range(len(class_labels))]
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=target_names))
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm, 
        annot=True, 
        fmt='d', 
        cmap='Blues',
        xticklabels=target_names,
        yticklabels=target_names
    )
    plt.title(f'{plant_name} - Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f'{config.MODELS_PATH}/{plant_name}_confusion_matrix.png', dpi=150)
    plt.show()
    
    return y_true, y_pred, predictions

# Evaluate all models
evaluation_results = {}

for plant_name, model_data in trained_models.items():
    y_true, y_pred, predictions = evaluate_model(
        model_data['model'],
        model_data['val_generator'],
        plant_name,
        model_data['class_indices']
    )
    evaluation_results[plant_name] = {
        'y_true': y_true,
        'y_pred': y_pred,
        'predictions': predictions
    }

## 8. Hierarchical Inference Pipeline

In [ ]:
class HierarchicalPlantDiseaseDetector:
    """
    Complete hierarchical pipeline for plant disease detection.
    
    Pipeline:
    1. Load and preprocess image
    2. Classify plant species using stage 1 model
    3. Load appropriate disease classifier
    4. Classify disease
    5. Return results with confidence scores
    """
    
    def __init__(self, plant_classifier_path, disease_models_path):
        """
        Initialize the hierarchical detector.
        
        Args:
            plant_classifier_path: Path to the stage 1 plant classifier model
            disease_models_path: Directory containing disease classifier models
        """
        self.img_size = config.IMG_SIZE
        self.disease_models_path = Path(disease_models_path)
        
        # Load plant classifier
        print("Loading plant classifier...")
        self.plant_classifier = keras.models.load_model(plant_classifier_path)
        
        # Plant classes (adjust based on your stage 1 model)
        self.plant_classes = ['Apple', 'Corn', 'Pepper', 'Potato', 'Tomato']
        
        # Cache for disease models (lazy loading)
        self.disease_models = {}
        self.disease_class_indices = {}
        
        print("✓ Hierarchical detector initialized")
    
    def load_disease_model(self, plant_name):
        """Lazy loads disease model for specific plant."""
        if plant_name not in self.disease_models:
            model_path = self.disease_models_path / f'{plant_name}_disease_classifier.h5'
            indices_path = self.disease_models_path / f'{plant_name}_class_indices.json'
            
            if not model_path.exists():
                raise FileNotFoundError(f"Disease model not found: {model_path}")
            
            print(f"Loading {plant_name} disease classifier...")
            self.disease_models[plant_name] = keras.models.load_model(str(model_path))
            
            with open(indices_path, 'r') as f:
                self.disease_class_indices[plant_name] = json.load(f)
        
        return self.disease_models[plant_name], self.disease_class_indices[plant_name]
    
    def preprocess_image(self, image_path):
        """Loads and preprocesses an image for prediction."""
        img = keras.preprocessing.image.load_img(
            image_path, 
            target_size=self.img_size
        )
        img_array = keras.preprocessing.image.img_to_array(img)
        img_array = img_array / 255.0  # Normalize
        img_array = np.expand_dims(img_array, axis=0)
        return img_array
    
    def predict(self, image_path, top_k=3):
        """
        Performs hierarchical prediction on an image.
        
        Args:
            image_path: Path to the input image
            top_k: Number of top predictions to return
        
        Returns:
            Dictionary containing:
            - plant: Detected plant species
            - plant_confidence: Confidence score for plant prediction
            - disease: Detected disease
            - disease_confidence: Confidence score for disease prediction
            - top_predictions: List of top k predictions
        """
        # Preprocess image
        img_array = self.preprocess_image(image_path)
        
        # Stage 1: Classify plant species
        plant_predictions = self.plant_classifier.predict(img_array, verbose=0)[0]
        plant_idx = np.argmax(plant_predictions)
        plant_name = self.plant_classes[plant_idx]
        plant_confidence = float(plant_predictions[plant_idx])
        
        # Stage 2: Classify disease
        disease_model, class_indices = self.load_disease_model(plant_name)
        disease_predictions = disease_model.predict(img_array, verbose=0)[0]
        
        # Get reverse mapping (index -> disease name)
        idx_to_disease = {v: k for k, v in class_indices.items()}
        
        # Get top k predictions
        top_indices = np.argsort(disease_predictions)[-top_k:][::-1]
        top_predictions = [
            {
                'disease': idx_to_disease[idx],
                'confidence': float(disease_predictions[idx])
            }
            for idx in top_indices
        ]
        
        # Main prediction
        disease_idx = top_indices[0]
        disease_name = idx_to_disease[disease_idx]
        disease_confidence = float(disease_predictions[disease_idx])
        
        return {
            'plant': plant_name,
            'plant_confidence': plant_confidence,
            'disease': disease_name,
            'disease_confidence': disease_confidence,
            'full_diagnosis': f'{plant_name}___{disease_name}',
            'top_predictions': top_predictions
        }
    
    def predict_batch(self, image_paths):
        """Performs hierarchical prediction on multiple images."""
        results = []
        for img_path in image_paths:
            try:
                result = self.predict(img_path)
                result['image_path'] = str(img_path)
                results.append(result)
            except Exception as e:
                results.append({
                    'image_path': str(img_path),
                    'error': str(e)
                })
        return results

# Initialize detector
detector = HierarchicalPlantDiseaseDetector(
    plant_classifier_path=config.PLANT_CLASSIFIER_PATH,
    disease_models_path=config.MODELS_PATH
)

## 9. Test Inference Pipeline

In [ ]:
def visualize_prediction(image_path, prediction_result):
    """Visualizes an image with its prediction results."""
    img = plt.imread(image_path)
    
    fig, ax = plt.subplots(1, 1, figsize=(10, 8))
    ax.imshow(img)
    ax.axis('off')
    
    # Create result text
    result_text = (
        f"Plant: {prediction_result['plant']} ({prediction_result['plant_confidence']:.2%})\n"
        f"Disease: {prediction_result['disease']} ({prediction_result['disease_confidence']:.2%})\n\n"
        f"Top {len(prediction_result['top_predictions'])} Predictions:\n"
    )
    
    for i, pred in enumerate(prediction_result['top_predictions'], 1):
        result_text += f"{i}. {pred['disease']}: {pred['confidence']:.2%}\n"
    
    ax.text(
        0.5, -0.1, result_text,
        transform=ax.transAxes,
        fontsize=11,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
        family='monospace'
    )
    
    plt.tight_layout()
    plt.show()

# Test on sample images
print("Testing inference pipeline on sample images...\n")

# Get sample images from each plant's validation set
sample_images = []
for plant in config.PLANT_DISEASES.keys():
    val_dir = Path(config.REORGANIZED_PATH) / plant / 'val'
    if val_dir.exists():
        # Get first image from first disease class
        for disease_dir in val_dir.iterdir():
            if disease_dir.is_dir():
                images = list(disease_dir.glob('*.jpg')) + list(disease_dir.glob('*.JPG'))
                if images:
                    sample_images.append(images[0])
                    break

# Run predictions
for img_path in sample_images[:3]:  # Test on first 3 samples
    print(f"\nPredicting: {img_path.name}")
    print("-" * 70)
    
    result = detector.predict(img_path)
    
    print(f"Plant: {result['plant']} (confidence: {result['plant_confidence']:.2%})")
    print(f"Disease: {result['disease']} (confidence: {result['disease_confidence']:.2%})")
    print(f"Full diagnosis: {result['full_diagnosis']}")
    print(f"\nTop 3 predictions:")
    for i, pred in enumerate(result['top_predictions'], 1):
        print(f"  {i}. {pred['disease']}: {pred['confidence']:.2%}")
    
    visualize_prediction(img_path, result)

## 10. Batch Inference Example

In [ ]:
# Example: Batch prediction on validation set
def batch_inference_demo(num_samples=10):
    """Demonstrates batch inference on multiple images."""
    print("=" * 70)
    print("BATCH INFERENCE DEMO")
    print("=" * 70)
    
    # Collect sample images
    sample_images = []
    for plant in config.PLANT_DISEASES.keys():
        val_dir = Path(config.REORGANIZED_PATH) / plant / 'val'
        if val_dir.exists():
            for disease_dir in val_dir.iterdir():
                if disease_dir.is_dir():
                    images = list(disease_dir.glob('*.jpg'))[:2]
                    sample_images.extend(images)
    
    sample_images = sample_images[:num_samples]
    
    # Batch prediction
    print(f"\nProcessing {len(sample_images)} images...\n")
    results = detector.predict_batch(sample_images)
    
    # Create results DataFrame
    df_results = pd.DataFrame([
        {
            'Image': Path(r['image_path']).name,
            'Plant': r['plant'],
            'Plant Conf.': f"{r['plant_confidence']:.2%}",
            'Disease': r['disease'],
            'Disease Conf.': f"{r['disease_confidence']:.2%}",
        }
        for r in results if 'error' not in r
    ])
    
    print(df_results.to_string(index=False))
    
    # Save results
    df_results.to_csv(f'{config.MODELS_PATH}/batch_inference_results.csv', index=False)
    print(f"\n✓ Results saved to: {config.MODELS_PATH}/batch_inference_results.csv")

batch_inference_demo()

## 11. Model Performance Summary

In [ ]:
def generate_performance_summary(trained_models, training_histories):
    """Generates a comprehensive performance summary for all models."""
    print("=" * 70)
    print("MODEL PERFORMANCE SUMMARY")
    print("=" * 70)
    
    summary_data = []
    
    for plant_name, model_data in trained_models.items():
        history = training_histories[plant_name]
        
        # Get best metrics
        best_val_acc = max(history['val_accuracy'])
        best_train_acc = max(history['accuracy'])
        final_val_loss = history['val_loss'][-1]
        num_classes = len(model_data['class_indices'])
        num_params = model_data['model'].count_params()
        
        summary_data.append({
            'Plant': plant_name,
            'Classes': num_classes,
            'Parameters': f"{num_params/1e6:.2f}M",
            'Best Val Acc': f"{best_val_acc:.4f}",
            'Best Train Acc': f"{best_train_acc:.4f}",
            'Final Val Loss': f"{final_val_loss:.4f}",
            'Epochs': len(history['accuracy'])
        })
    
    df_summary = pd.DataFrame(summary_data)
    print("\n", df_summary.to_string(index=False))
    
    # Save summary
    df_summary.to_csv(f'{config.MODELS_PATH}/model_performance_summary.csv', index=False)
    print(f"\n✓ Summary saved to: {config.MODELS_PATH}/model_performance_summary.csv")
    
    # Visualize comparative performance
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    plants = [d['Plant'] for d in summary_data]
    val_accs = [float(d['Best Val Acc']) for d in summary_data]
    num_classes = [d['Classes'] for d in summary_data]
    
    # Validation accuracy comparison
    axes[0].bar(plants, val_accs, color='steelblue', alpha=0.7)
    axes[0].set_title('Best Validation Accuracy by Plant')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_ylim([0.8, 1.0])
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Number of classes
    axes[1].bar(plants, num_classes, color='coral', alpha=0.7)
    axes[1].set_title('Number of Disease Classes by Plant')
    axes[1].set_ylabel('Number of Classes')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(f'{config.MODELS_PATH}/performance_comparison.png', dpi=150)
    plt.show()

generate_performance_summary(trained_models, training_histories)

## 12. Export for Production

In [ ]:
# Save complete configuration and metadata
metadata = {
    'system': 'Hierarchical Plant Disease Detection',
    'version': '2.0',
    'created': pd.Timestamp.now().isoformat(),
    'architecture': 'MobileNetV3Large',
    'image_size': config.IMG_SIZE,
    'plants': list(config.PLANT_DISEASES.keys()),
    'models': {
        plant: {
            'num_classes': len(config.PLANT_DISEASES[plant]),
            'classes': config.PLANT_DISEASES[plant],
            'model_path': f'{plant}_disease_classifier.h5',
            'class_indices_path': f'{plant}_class_indices.json'
        }
        for plant in config.PLANT_DISEASES.keys()
    }
}

with open(f'{config.MODELS_PATH}/system_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("=" * 70)
print("PRODUCTION EXPORT COMPLETE")
print("=" * 70)
print(f"\nAll models and metadata saved to: {config.MODELS_PATH}")
print("\nFiles generated:")
for plant in config.PLANT_DISEASES.keys():
    print(f"  • {plant}_disease_classifier.h5")
    print(f"  • {plant}_class_indices.json")
print("  • system_metadata.json")
print("  • model_performance_summary.csv")
print("  • Training history plots")
print("  • Confusion matrices")
print("\n✓ System ready for deployment!")

## 13. Usage Instructions

In [ ]:
usage_instructions = """
═══════════════════════════════════════════════════════════════════════
HIERARCHICAL PLANT DISEASE DETECTION SYSTEM - USAGE GUIDE
═══════════════════════════════════════════════════════════════════════

1. SINGLE IMAGE PREDICTION:
   ────────────────────────────────────────────────────────────────
   detector = HierarchicalPlantDiseaseDetector(
       plant_classifier_path='path/to/plant_classifier.h5',
       disease_models_path='path/to/disease_models/'
   )
   
   result = detector.predict('path/to/image.jpg')
   print(f"Plant: {result['plant']}")
   print(f"Disease: {result['disease']}")
   print(f"Confidence: {result['disease_confidence']:.2%}")

2. BATCH PREDICTION:
   ────────────────────────────────────────────────────────────────
   image_paths = ['img1.jpg', 'img2.jpg', 'img3.jpg']
   results = detector.predict_batch(image_paths)
   
   for result in results:
       print(f"{result['image_path']}: {result['full_diagnosis']}")

3. CUSTOM INFERENCE:
   ────────────────────────────────────────────────────────────────
   # Load specific disease model
   model = keras.models.load_model('Apple_disease_classifier.h5')
   
   # Preprocess image
   img = keras.preprocessing.image.load_img('apple.jpg', target_size=(224, 224))
   img_array = keras.preprocessing.image.img_to_array(img) / 255.0
   img_array = np.expand_dims(img_array, axis=0)
   
   # Predict
   predictions = model.predict(img_array)

4. API INTEGRATION EXAMPLE:
   ────────────────────────────────────────────────────────────────
   from flask import Flask, request, jsonify
   
   app = Flask(__name__)
   detector = HierarchicalPlantDiseaseDetector(...)
   
   @app.route('/predict', methods=['POST'])
   def predict():
       file = request.files['image']
       result = detector.predict(file)
       return jsonify(result)

5. MODEL OPTIMIZATION:
   ────────────────────────────────────────────────────────────────
   # Convert to TensorFlow Lite for mobile deployment
   converter = tf.lite.TFLiteConverter.from_keras_model(model)
   converter.optimizations = [tf.lite.Optimize.DEFAULT]
   tflite_model = converter.convert()
   
   with open('model.tflite', 'wb') as f:
       f.write(tflite_model)

═══════════════════════════════════════════════════════════════════════
PERFORMANCE TIPS:
═══════════════════════════════════════════════════════════════════════
• Models are loaded lazily - only when needed
• Use batch prediction for multiple images to amortize loading time
• For production, consider converting to TFLite or ONNX
• Image preprocessing is automatically handled
• Confidence scores indicate prediction reliability

═══════════════════════════════════════════════════════════════════════
"""

print(usage_instructions)

# Save to file
with open(f'{config.MODELS_PATH}/USAGE_GUIDE.txt', 'w') as f:
    f.write(usage_instructions)